# 04 Findings


Summary of the EDA of the Diabetes 130-US Hospitals dataset (1999-2008).
Every number below is computed in this notebook from `data/raw/` and `data/processed/`.
Details: `01_data_overview`, `02_cleaning`, `03_univariate_bivariate`.


## Question

Which patient, admission and treatment characteristics go with readmission to hospital within 30 days
(`readmitted_30`, from `readmitted == "<30"`)?


In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from eda_diabetes.data import load_processed, load_raw
from eda_diabetes.paths import REPORTS_FIGURES

sns.set_theme(style="whitegrid")
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)

raw = load_raw()
df = load_processed()
TARGET = "readmitted_30"


def rate_table(col, bins=None, labels=None) -> pd.DataFrame:
    key = pd.cut(df[col], bins, labels=labels) if bins is not None else df[col]
    out = df.groupby(key, observed=True)[TARGET].agg(rate="mean", n="size")
    return out.round(3)

## 1. Data and cohort


In [ ]:
print(f"raw encounters: {len(raw):,} ({raw['patient_nbr'].nunique():,} patients)")
print(f"cohort after cleaning: {len(df):,} (one encounter per patient)")
print(f"raw columns: {raw.shape[1]}, processed columns: {df.shape[1]}")
pd.DataFrame(
    {
        "raw": raw["readmitted"].value_counts(normalize=True),
        "cohort": df["readmitted"].value_counts(normalize=True),
    }
).round(3)

- The raw file has 101,766 encounters of 71,518 patients, so rows are not independent.
- The cohort keeps the first encounter of each patient and removes expired and hospice discharges (cannot be readmitted) and 3 unknown-gender rows.
  This leaves 69,987 rows.
- The target is imbalanced: 9.0% of the cohort is readmitted within 30 days (11.2% before cleaning).


## 2. Data quality


In [ ]:
quality = pd.Series(
    {
        "weight missing": raw["weight"].isna().mean(),
        "medical_specialty missing": raw["medical_specialty"].isna().mean(),
        "payer_code missing": raw["payer_code"].isna().mean(),
        "max_glu_serum not measured": raw["max_glu_serum"].isna().mean(),
        "A1Cresult not measured": raw["A1Cresult"].isna().mean(),
        "race missing": raw["race"].isna().mean(),
    },
    name="share of raw rows",
)
(quality * 100).round(1).to_frame()

- `weight` is 97% missing and was dropped. `medical_specialty` (49%) and `payer_code` (40%) are kept with an `Unknown` level.
- For the two lab columns NaN means the test was not done, so it is a real category (`None`), not a gap to fill.
- Some ID codes mean missing (`NULL`, `Not Mapped`). They stay NaN in the processed ID columns.
- Four medication columns are constant or nearly so and were dropped.


## 3. What goes with 30-day readmission


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
panels = [
    ("discharge_group", rate_table("discharge_group").query("n >= 100").sort_values("rate")),
    (
        "number_inpatient (prior year)",
        rate_table("number_inpatient", [-1, 0, 1, 2, 99], ["0", "1", "2", "3+"]),
    ),
    (
        "time_in_hospital (days)",
        rate_table("time_in_hospital", [0, 2, 4, 7, 14], ["1-2", "3-4", "5-7", "8-14"]),
    ),
    ("age", rate_table("age")),
]
for ax, (title, table) in zip(axes.flat, panels, strict=True):
    ax.bar(table.index.astype(str), table["rate"])
    ax.axhline(df[TARGET].mean(), color="k", ls="--", lw=1)
    ax.set_title(title)
    ax.set_ylabel("P(readmitted < 30 days)")
    ax.tick_params(axis="x", rotation=45)
fig.suptitle("30-day readmission rate (dashed: overall 9.0%)")
fig.tight_layout()
fig.savefig(REPORTS_FIGURES / "findings_main_drivers.png", dpi=150, bbox_inches="tight")

In [ ]:
rate_table("discharge_group")

In [ ]:
rate_table("number_inpatient", [-1, 0, 1, 2, 99], ["0", "1", "2", "3+"])

In [ ]:
rate_table("time_in_hospital", [0, 2, 4, 7, 14], ["1-2", "3-4", "5-7", "8-14"])

1. **Discharge destination.** Patients discharged to a facility (SNF, rehab, other inpatient care) are readmitted at 15.2%, against 7.4% for patients discharged home. This is the strongest categorical association (Cramer's V 0.108).
2. **Prior inpatient visits.** The rate rises with each visit in the previous year: 8.1% with none, 12.9% with one, 18.5% with two, 26.5% with three or more. Few patients are in the high groups (896 with three or more).
3. **Length of stay.** The rate rises from 7.2% for 1-2 days to 11.8% for 8-14 days.
4. **Age.** Rates are low below 60, then rise to about 10-11% at ages 70-90. Groups under 20 have fewer than 700 patients combined.
5. **Treatment.** Patients on diabetes medication are readmitted more (9.4% vs 7.6%), as are those with a medication change (9.4% vs 8.6%). Patients on insulin dose `Down` or `Up` are higher than `No`. These gaps are small and are likely markers of sicker patients, not effects of the drugs.


## 4. What does not


In [ ]:
rate_table("A1Cresult")

- **HbA1c result** (`A1Cresult`, chi-square p = 0.061) and **max glucose serum** (p = 0.076) show no clear association at 5%. Rates are 8.2-9.1% across levels.
- **Gender** has no association (p = 0.59).
- **Race, admission type and admission source** have Cramer's V of 0.011-0.016.
- Strack et al. (2014) reported that measuring HbA1c goes with lower readmission. This cohort does not show a clear A1C effect.
  The cohort and method differ (one encounter per patient here), so this is not a direct test of that paper.


## 5. Effect sizes are small


The largest effects belong to prior inpatient visits, discharge destination and length of stay, and all are modest.
For numeric columns the probability that a random readmitted patient has a higher value than a random other patient is at most 0.56 (`time_in_hospital`).
No Cramer's V is above 0.11. Expect a model on these features to have limited discrimination.


## 6. Outliers


Visit counts are zero-inflated and heavy-tailed (87-93% zeros, maximum 42 visits in a year).
The extremes are kept: they look like real heavy use, and the tail carries signal (the 30-day rate above the 99th percentile of `number_inpatient` is about 29%).


## 7. Limitations

- **Old data.** 1999-2008, from 130 hospitals. Practice and coding have changed since then.
- **Selection.** Only encounters of diabetic patients with a stay of 1-14 days.
- **Not causal.** Observational data. Treatment variables mark sicker patients.
- **First encounter only.** Keeping one encounter per patient avoids dependence but drops later visits. Patients with many admissions are under-represented.
- **Outcome definition.** `readmitted` records readmission to the same system. Readmissions elsewhere, and deaths after discharge, are not seen.
- **Missing data.** Nearly half of `medical_specialty` is `Unknown`. Hidden missing codes in the ID columns were set to NaN or `Unknown`.
- **Grouping choices.** The discharge, admission-source and ICD-9 groupings are my own judgement (`src/eda_diabetes/data.py`). Other groupings may change the results.


## 8. Recommendations for modelling

- **Target:** `readmitted_30`. Keep the 9% base rate in mind. Use stratified splits, and report PR-AUC and recall next to ROC-AUC.
- **Split by patient.** The cohort has one row per patient, so this is already satisfied. If the full data is used, group by `patient_nbr`.
- **Features to start with:** `discharge_group`, `number_inpatient`, `time_in_hospital`, `age`, `number_diagnoses`, `num_medications`, `n_meds_used`, `diabetesMed`, `insulin`, `diag_1`.
- **Redundant features:** use `total_visits` or its three parts, not both, in a linear model.
- **Transforms:** `log1p` or tree-based models for the visit counts. One-hot or target encode the categoricals, with the rare levels merged.
- **Check for leakage.** `discharge_group` is known at discharge, which is the prediction time of a readmission model. Confirm this matches the intended use.


## 9. Figures

All figures are in `reports/figures/`:
`numeric_distributions`, `categorical_distributions`, `numeric_boxplots`, `target_rate_by_category`, `numeric_by_target`,
`spearman_heatmap`, `cramers_v_heatmap`, `a1c_by_change`, `prior_inpatient_by_age` (from notebook 03) and `findings_main_drivers` (this notebook).
